# CAG exploration chatbot procédures

In [1]:
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pathlib import Path

In [2]:
def check_dir_exists(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Le chemin n'existe pas : {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"Le chemin n'est pas un dossier : {path}")
    
check_dir_exists(Path.cwd() / ".." / "data" / "files_test")


## PDF METHOD

In [3]:
# chargement du répertoire des PDF

pdfs = Path.cwd() / ".." / "data" / "files_test"

if not pdfs.exists():
    raise FileNotFoundError(f"Chemin introuvable: {pdfs}")
if not pdfs.is_dir():
    raise NotADirectoryError(f"Ce n'est pas un dossier: {pdfs}")

def load_docs(doc_dir: Path):
    docs = []

    doc_paths = sorted(
        list(doc_dir.rglob("*.pdf")) + list(doc_dir.rglob("*.docx")),
        key=lambda p: str(p).lower()
    )

    for path in doc_paths:
        if path.suffix.lower() == ".pdf":
            loader = PyPDFLoader(str(path))
        elif path.suffix.lower() == ".docx":
            loader = Docx2txtLoader(str(path))
        else:
            continue

        pages = loader.load()

        for d in pages:
            d.metadata["source_path"] = str(path)
            d.metadata["source_file"] = path.name
            d.metadata["file_type"] = path.suffix.lower()

        docs.extend(pages)

    return docs

raw_docs = load_docs(pdfs)
print(raw_docs[0])

page_content='CREATION DE COMPTE 
 
 
A réception de ce message, dans votre messagerie : 
 
 
Ouvrez-le et Cliquez sur Se connecter à Zoom 
 
 
Remplissez les 4 zones demandées : prénom, nom de famille, mot de passe, confirmation du mot de 
passe, puis cliquez sur Continuer' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-06-28T11:22:33+02:00', 'author': 'Frédéric ARNAUD365', 'moddate': '2023-06-28T11:22:33+02:00', 'source': '/home/sachard/mairie/backend/../data/files_test/1 - Créer son Compte.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_path': '/home/sachard/mairie/backend/../data/files_test/1 - Créer son Compte.pdf', 'source_file': '1 - Créer son Compte.pdf', 'file_type': '.pdf'}


In [4]:
from IPython.display import display, Markdown

def show_doc(doc):
    md = f"""
### Source
- Fichier : `{doc.metadata.get('source_file')}`
- Page : `{doc.metadata.get('page')}`

---

{doc.page_content}
"""
    display(Markdown(md))

show_doc(raw_docs[2])


### Source
- Fichier : `2 - Installer l'Application.pdf`
- Page : `0`

---

Installation de l’ Application 
Sur PC 
 
Ouvrez l’Explorateur de Fichiers, puis l’Emplacement CASTELNAU - COMMUN (X :) 
 
 
 
Ensuite, ouvrez successivement les dossiers Informatique, puis Utilitaires et enfin Zoom 
Vous arrivez sur cette interface : 
 
 
 
Maintenant, double-cliquez sur ZoomInstaller.exe pour débuter l’installation de l’ Application Zoom  
 
Sur la fenêtre qui s’ouvre, cliquez sur Exécuter


## CHUNKING, EMBEDDINGS

In [5]:
raw_docs = load_docs(pdfs)

# Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=120,
)

chunks = splitter.split_documents(raw_docs)
print("Chunks:", len(chunks))


Chunks: 201


In [6]:
#test API ollama

llm = ChatOllama(
    model="mistral",
    base_url="http://localhost:11434",
    temperature=0.2,
)

In [7]:
lengths = [len(d.page_content) for d in chunks]
max_len = max(lengths)
idx = lengths.index(max_len)

print("Nb chunks:", len(chunks))
print("Max chars:", max_len)
print("Index chunk max:", idx)
print("Metadata:", chunks[idx].metadata)
print("Aperçu:", chunks[idx].page_content[:500])

Nb chunks: 201
Max chars: 498
Index chunk max: 80
Metadata: {'producer': 'Microsoft® Word pour Microsoft\xa0365', 'creator': 'Microsoft® Word pour Microsoft\xa0365', 'creationdate': '2025-03-21T11:47:51+01:00', 'author': 'MATHIEU RESPAUT', 'moddate': '2025-03-21T11:47:51+01:00', 'source': '/home/sachard/mairie/backend/../data/files_test/Charte informatique 03-2025.pdf', 'total_pages': 18, 'page': 8, 'page_label': '9', 'source_path': '/home/sachard/mairie/backend/../data/files_test/Charte informatique 03-2025.pdf', 'source_file': 'Charte informatique 03-2025.pdf', 'file_type': '.pdf'}
Aperçu: est imposé de ne pas les relayer. II ne doit donc pas en solliciter l'envoi en participant à des groupes de 
discussion, ou en consultant des sites, dont le caractère est proscrit, qui pourrait enregistrer ses 
coordonnées. 
 Utilisation de la messagerie électronique à des fins personnelles 
Il est considéré que tout message reçu ou envoyé à partir du poste de travail mis à la disposition de 
l'uti

In [9]:
#embeddings
chunks = splitter.split_documents(raw_docs)
texts = [doc.page_content for doc in chunks]

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434",
)

#indexation FAISS

db = None
batch_size = 16

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    if db is None:
        db = FAISS.from_documents(batch, embeddings)
    else:
        db.add_documents(batch)

db.save_local("./faiss_index")

# Chargement de l'index FAISS
db = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [10]:
print(len(embeddings.embed_query("Bonjour")))
print(len(chunks))

768
201


## DEBUG

In [11]:
# debug retrieval RAG

def debug_retrieval(db, question, k=5):
    docs = db.similarity_search(question, k=k)
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source_file") or d.metadata.get("source")
        page = d.metadata.get("page")
        print(f"\n--- {i}. {src} page {page} ---")
        print(d.page_content[:500])
    return docs
debug_retrieval(db, "comment se connecter au wifi résidents ?", k=5)


--- 1. 1 - Créer son Compte.pdf page 0 ---
CREATION DE COMPTE 
 
 
A réception de ce message, dans votre messagerie : 
 
 
Ouvrez-le et Cliquez sur Se connecter à Zoom 
 
 
Remplissez les 4 zones demandées : prénom, nom de famille, mot de passe, confirmation du mot de 
passe, puis cliquez sur Continuer

--- 2. ZOOM - Tuto.pdf page 1 ---
CREATION DE COMPTE 
 
 
A réception de ce message, dans votre messagerie : 
 
 
Ouvrez-le et Cliquez sur Se connecter à Zoom 
 
 
Remplissez les 4 zones demandées : prénom, nom de famille, mot de passe, confirmation du mot de 
passe, puis cliquez sur Continuer

--- 3. Charte informatique 03-2025.pdf page 9 ---
l'accord préalable de la DSI ; 
• De connecter à son poste informatique des équipements personnels : smartphones, tablettes, 
clé USB, etc. ; 
• D’installer des logiciels ou applications externes sans l’accord préalable de la DSI.

--- 4. Wifi Mairie - Procédure Connexion.pdf page 3 ---
W I F I     M A I R I E    
  
DIRECTION DU SYSTEME 
D’INFOR

[Document(id='f2438ca9-539d-4a07-9e2e-492e29bcb743', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-06-28T11:22:33+02:00', 'author': 'Frédéric ARNAUD365', 'moddate': '2023-06-28T11:22:33+02:00', 'source': '/home/sachard/mairie/backend/../data/files_test/1 - Créer son Compte.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_path': '/home/sachard/mairie/backend/../data/files_test/1 - Créer son Compte.pdf', 'source_file': '1 - Créer son Compte.pdf', 'file_type': '.pdf'}, page_content='CREATION DE COMPTE \n \n \nA réception de ce message, dans votre messagerie : \n \n \nOuvrez-le et Cliquez sur Se connecter à Zoom \n \n \nRemplissez les 4 zones demandées : prénom, nom de famille, mot de passe, confirmation du mot de \npasse, puis cliquez sur Continuer'),
 Document(id='4ddbf413-c452-46ba-bf5f-1b4ea1f88774', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-07-05T16:20:37+

## EVAL

In [13]:
# eval retrieval RAG

question = "comment se connecter au wifi résidents ?"

eval_set = [
  {
    "question": "comment obtenir le code pour se connecter au wifi résidents ?",
    "gold_snippet": "demander à l'accueil de l'ehpad",
  },
  {
    "question": "que dois je faire sur téléphone ou tablette ?",
    "gold_snippet": "désactiver l'option 'utiliser une adresse mac aléatoire'",
  },
]

def recall_at_k(db, eval_set, k=5):
    hits = 0
    for e in eval_set:
        docs = db.similarity_search(e["question"], k=k)
        combined = " ".join(d.page_content.lower() for d in docs)
        if e["gold_snippet"].lower() in combined:
            hits += 1
    return hits / len(eval_set)

print("Recall@5:", recall_at_k(db, eval_set, k=5))
print("Recall@10:", recall_at_k(db, eval_set, k=10))

Recall@5: 0.0
Recall@10: 0.0


In [ ]:
question = "comment se connecter au wifi résidents ?"
for doc in db.similarity_search(question, k=5):
    print(doc.metadata.get("source"), doc.metadata.get("page"))
    print(doc.page_content[:200])

/mnt/u/pdfs/Wifi Mairie - Procédure Connexion.pdf 1
W I F I     M A I R I E    
  
DIRECTION DU SYSTEME 
D’INFORMATION  WIFI MAIRIE  
  
  
  
5. Dans cette nouvelle fenêtre, cochez se connecter automatiquement et cliquez sur Se connecter  
 
  
  
  

/mnt/u/pdfs/Wifi Mairie - Procédure Connexion.pdf 4
W I F I     M A I R I E    
  
DIRECTION DU SYSTEME 
D’INFORMATION  WIFI MAIRIE  
  
Une fois le PC redémarré,   
  
  
Vous obtenez cette fenêtre  
  
Tout est Ok, vous êtes bien connecté au WIFI_MAI
/mnt/u/pdfs/Wifi Mairie - Procédure Connexion.pdf 0
W I F I     M A I R I E    
  
DIRECTION DU SYSTEME 
D’INFORMATION  WIFI MAIRIE  
PROCEDURE DE CONNEXION   
 
 
1. Allumez votre ordinateur (sur votre session utilisateur)  
  
2. Cliquez sur l’icône 
/mnt/u/pdfs/Wifi Mairie - Procédure Connexion.pdf 3
W I F I     M A I R I E    
  
DIRECTION DU SYSTEME 
D’INFORMATION  WIFI MAIRIE  
  
Vous revenez sur la fenêtre précédente, cliquez de nouveau sur Se connecter   
 
  
  
  
Cette fenêtre 

## TESTS

In [ ]:
# prompt
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant qui répond UNIQUEMENT à partir du contexte.
Si la réponse n'est pas dans le contexte, dis "Je ne sais pas".

Contexte:
{context}

Question:
{question}

Réponse (sans références):
""")

#Test
query = "Comment me connecter au wifi résidents?"
results = db.similarity_search(query, k=5)

# for i, doc in enumerate(results, start=1):
#     src = doc.metadata.get("source")
#     page = doc.metadata.get("page")
#     print(f"\n--- Résultat {i} ({src}, page {page}) ---")
#     print(doc.page_content[:400])

# Retriever
retriever = db.as_retriever(search_kwargs={"k": 5})

# Chaîne RAG
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

result = rag_chain.invoke("Comment me connecter au wifi résidents?")
print(result.content)


 Pour se connecter au WiFi des résidents, suivez les étapes suivantes :

1. Allumez votre ordinateur (sur votre session utilisateur).
2. Cliquez sur l’icône du Wifi dans la barre des tâches (en bas à droite de l’écran).
3. Dans la liste qui apparait, cliquez sur WiFi_RESIDENTS.
4. Cliquer sur afficher plus.
5. Sélectionner Adresse Mac.
6. Renseigner le code du téléphone.


In [ ]:
# prompt
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant qui répond UNIQUEMENT à partir du contexte.
Si la réponse n'est pas dans le contexte, dis "Je ne sais pas".

Contexte:
{context}

Question:
{question}

Réponse (sans références):
""")

#Test
query = "Quelle est l'étape 7 de la procédure indiquée ?"
results = db.similarity_search(query, k=5)

for i, doc in enumerate(results, start=1):
    src = doc.metadata.get("source")
    page = doc.metadata.get("page")
    print(f"\n--- Résultat {i} ({src}, page {page}) ---")
    print(doc.page_content[:400])

# Retriever
retriever = db.as_retriever(search_kwargs={"k": 5})

# Chaîne RAG
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

result = rag_chain.invoke("Comment me connecter au wifi mairie?")
print(result.content)


--- Résultat 1 (/mnt/u/pdfs/Procédure connexion WiFI_RESIDENTS.pdf, page 0) ---
AIDE CONNEXION AU WIFI_RESIDENTS 
1ére étape : demander un code auprès de l’accueil de l’Ehpad 
 
Pour information, ce code est à usage unique. Cela signifie qu’il ne pourra être utilisé que sur 1 seul 
équipement (téléphone, tablette, ordinateur, cadre photo…) 
 
Si vous disposez de plusieurs équipements, il faudra donc demander plusieurs codes. 
 
Ce code est sans limitation de temps et de durée

--- Résultat 2 (/mnt/u/pdfs/Procédure connexion WiFI_RESIDENTS.pdf, page 1) ---
- Sur un ordinateur ou tout autre équipement 
 
1- Sélectionner le réseau WiFI_RESIDENTS              2- Renseigner le code 
 
 
 
 
 
 
 
 
 
 
/!\ En cas de problème, merci de voir avec l’accueil de l’EHPAD qui escaladera l’incident.

--- Résultat 3 (/mnt/u/pdfs/Wifi Mairie - Procédure Connexion.pdf, page 0) ---
W I F I     M A I R I E    
  
DIRECTION DU SYSTEME 
D’INFORMATION  WIFI MAIRIE  
PROCEDURE DE CONNEXION   
 
 
1. Allume